In [1]:
import ScraperFC as sfc 
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

import csv
import pickle

In [3]:
sofascore= sfc.Sofascore()


In [ ]:
# SCRAPEO URLS PARTIDOS

driver = webdriver.Chrome()
driver.get("https://www.sofascore.com/es-la/torneo/futbol/spain/laliga/8#id:77559,tab:matches")

wait = WebDriverWait(driver, 15)

# 1. Cerrar pop-up de "Consentir"
try:
    consentir_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Consentir')]")))
    consentir_btn.click()
except:
    pass

# 2. Cerrar pop-up de "CONFIRMAR" o botón de cierre
try:
    cerrar_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="portals"]//button')))
    cerrar_btn.click()
except:
    pass

time.sleep(3)

# 3. Retroceder hasta la jornada 1
while True:
    try:
        prev_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="tabpanel-round"]/div/div[1]/div/button[1]')))
        driver.execute_script("arguments[0].click();", prev_btn)
        time.sleep(2)
    except:
        break

# 4. Recorrer jornadas hacia adelante y extraer URLs
urls = []
while True:
    partidos = driver.find_elements(By.CSS_SELECTOR, "a[class^='event-hl-']")
    for p in partidos:
        href = p.get_attribute("href")
        if href and href not in urls:
            urls.append(href)

    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="tabpanel-round"]/div/div[1]/div/button[2]')))
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(2)
    except:
        break

# --- Eliminar una URL específica ---
url_descartar = "https://www.sofascore.com/es-la/football/match/getafe-osasuna/vgbsjhb#id:14083640"
if url_descartar in urls:
    urls.remove(url_descartar)

# --- Empaquetar en lista de listas (38 jornadas de 10 partidos) ---
jornadas = [urls[i:i+10] for i in range(0, len(urls), 10)]

driver.quit()

# Ahora 'jornadas' contiene las 38 sublistas con las URLs

In [ ]:


# Supongamos que ya tienes la variable 'jornadas' creada como lista de listas
# jornadas = [
#   ["url_jornada1_partido1", ..., "url_jornada1_partido10"],
#   ["url_jornada2_partido1", ..., "url_jornada2_partido10"],
#   ...
# ]

with open("partidos_laliga.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    # Opcional: encabezado
    writer.writerow(["Jornada", "Partido", "URL"])
    
    for jornada_num, jornada in enumerate(jornadas, start=1):
        for partido_num, url in enumerate(jornada, start=1):
            writer.writerow([jornada_num, partido_num, url])

print("📂 Archivo 'jornadas_partidos.csv' creado con éxito")

📂 Archivo 'jornadas_partidos.csv' creado con éxito


In [4]:
df_url=pd.read_csv('partidos_laliga.csv')
df_url


,Jornada,Partido,URL
0,1,1,https://www.sofascore.com/es-la/football/match...
1,1,2,https://www.sofascore.com/es-la/football/match...
2,1,3,https://www.sofascore.com/es-la/football/match...
3,1,4,https://www.sofascore.com/es-la/football/match...
4,1,5,https://www.sofascore.com/es-la/football/match...
...,...,...,...
375,38,6,https://www.sofascore.com/es-la/football/match...
376,38,7,https://www.sofascore.com/es-la/football/match...
377,38,8,https://www.sofascore.com/es-la/football/match...
378,38,9,https://www.sofascore.com/es-la/football/match...


In [ ]:
## SCRAPEO PARIDOS STATS 
import pandas as pd
import re

# Asegurar que Jornada es numérica y ordenar
df_url["Jornada"] = pd.to_numeric(df_url["Jornada"], errors="coerce")
df_url = df_url.dropna(subset=["Jornada", "URL"]).sort_values(["Jornada", "Partido"])

# Filtrar solo jornadas 0 a 14
df_url_filtrado = df_url[df_url["Jornada"].between(0, 14)].drop_duplicates(subset=["Jornada", "Partido", "URL"])

# Crear un ExcelWriter
with pd.ExcelWriter("jornadas_00_14_stats.xlsx", engine="xlsxwriter") as writer:
    for j in sorted(df_url_filtrado["Jornada"].unique()):
        urls_jornada = df_url_filtrado.loc[df_url_filtrado["Jornada"] == j, ["Partido", "URL"]]

        df_partidos = []

        for _, row in urls_jornada.iterrows():
            partido_num = int(row["Partido"]) if pd.notnull(row["Partido"]) else None
            url = row["URL"]

            # Extraer el ID de la URL (después de '#id:')
            match = re.search(r"id:(\d+)", url)
            partido_id = match.group(1) if match else None

            # Tu lógica de scraping intacta
            df = sofascore.scrape_team_match_stats(url)

            # Añadir metadatos
            df["Jornada"] = j
            df["Partido"] = partido_num
            df["id"] = partido_id

            df_partidos.append(df)

            print(f"\nJornada {j} - Partido {partido_num} (ID {partido_id}):")
            print(df.head())

        # Concatenar todos los partidos de la jornada y escribir en su hoja
        if df_partidos:
            df_jornada = pd.concat(df_partidos, ignore_index=True)
            hoja = f"Jornada_{j:02d}"
            df_jornada.to_excel(writer, sheet_name=hoja, index=False)
            print(f"Guardado hoja: {hoja}")


📊 Jornada 1 - Partido 1 (ID 14082854):
               name  home  away  compareCode statisticsType valueType  \
0   Ball possession   44%   56%            2       positive     event   
1    Expected goals  0.56  3.43            2       positive     event   
2       Big chances     1     4            2       positive     event   
3       Total shots     7    16            2       positive     event   
4  Goalkeeper saves     2     1            1       positive     event   

   homeValue  awayValue  renderType               key period           group  \
0      44.00      56.00           2    ballPossession    ALL  Match overview   
1       0.56       3.43           1     expectedGoals    ALL  Match overview   
2       1.00       4.00           1  bigChanceCreated    ALL  Match overview   
3       7.00      16.00           1  totalShotsOnGoal    ALL  Match overview   
4       2.00       1.00           1   goalkeeperSaves    ALL  Match overview   

   homeTotal  awayTotal  Jornada  Partid

In [45]:
sofascore.get_match_dict('https://www.sofascore.com/es-la/football/match/girona-fc-rayo-vallecano/tgbsoKj#id:14082854')

{'correctAiInsight': True,
 'correctHalftimeAiInsight': True,
 'tournament': {'name': 'LaLiga',
  'slug': 'laliga',
  'category': {'name': 'Spain',
   'slug': 'spain',
   'sport': {'name': 'Football', 'slug': 'football', 'id': 1},
   'id': 32,
   'country': {'alpha2': 'ES',
    'alpha3': 'ESP',
    'name': 'Spain',
    'slug': 'spain'},
   'flag': 'spain',
   'alpha2': 'ES',
   'fieldTranslations': {'nameTranslation': {'ar': 'إسبانيا',
     'hi': 'स्पेन',
     'bn': 'স্পেন'},
    'shortNameTranslation': {}}},
  'uniqueTournament': {'name': 'LaLiga',
   'slug': 'laliga',
   'primaryColorHex': '#2f4a89',
   'secondaryColorHex': '#f4a32e',
   'category': {'name': 'Spain',
    'slug': 'spain',
    'sport': {'name': 'Football', 'slug': 'football', 'id': 1},
    'id': 32,
    'country': {'alpha2': 'ES',
     'alpha3': 'ESP',
     'name': 'Spain',
     'slug': 'spain'},
    'flag': 'spain',
    'alpha2': 'ES',
    'fieldTranslations': {'nameTranslation': {'ar': 'إسبانيا',
      'hi': 'स्पेन',

In [ ]:
## SCRAPEO PARIDOS INFO  
import pandas as pd
import re
import xlsxwriter 
import datetime # Necesaria para convertir el Timestamp Unix
import time

# ==============================================================================
# FUNCIÓN EXTERNA DE SOFASCORE (PLACEHOLDER)
#
# Debes asegurarte de que tu librería 'sofascore' esté importada y que la 
# función 'get_match_dict(url)' esté disponible.
# El siguiente bloque es solo un ejemplo de simulación. REEMPLÁZALO
# con la importación y configuración de tu librería real.
#
# Ejemplo de Importación Real (Descomentar y usar):
# import sofascore
#
# Si no usas una librería, y solo tienes acceso a la función, asegúrate de que esté en tu entorno.

class SofascoreSim:
    """Clase de simulación para hacer el código ejecutable como ejemplo."""
    def get_match_dict(self, url):
        time.sleep(0.1) # Simular latencia
        match_id_match = re.search(r"id:(\d+)", url)
        if not match_id_match:
            return {}
        match_id = match_id_match.group(1)
        
        # Simulación de datos JSON con startTimestamp
        return {
            'id': int(match_id),
            'homeTeam': {'name': 'Local ' + match_id},
            'awayTeam': {'name': 'Visitante ' + match_id},
            'venue': {'name': 'Estadio Sim', 'capacity': 50000, 'venueCoordinates': {'latitude': 40.0, 'longitude': -3.0}},
            # Timestamp de ejemplo (15/08/2025 19:00:00 UTC)
            'startTimestamp': 1755277200, 
            'homeScore': {'current': 2},
            'awayScore': {'current': 1},
        }

# Asegúrate de usar tu implementación real aquí:
# sofascore = SofascoreSim() # Línea de simulación
# ==============================================================================


## 1. Función de Extracción de Datos (JSON) con Conversión de Timestamp

def extract_match_info_from_json(match_data: dict) -> dict:
    """
    Extrae y formatea los campos de datos solicitados, incluyendo la conversión 
    del Timestamp Unix a Fecha y Hora legibles.
    """
    if not match_data:
        return None

    # Extracción de campos
    partido_id = match_data.get('id')
    local = match_data.get('homeTeam', {}).get('name')
    visitante = match_data.get('awayTeam', {}).get('name')
    
    venue = match_data.get('venue', {})
    estadio = venue.get('name')
    capacidad = venue.get('capacity')
    
    coords = venue.get('venueCoordinates', {})
    latitud = coords.get('latitude')
    longitud = coords.get('longitude')

    home_score = match_data.get('homeScore', {}).get('current')
    away_score = match_data.get('awayScore', {}).get('current')
    
    if home_score is not None and away_score is not None:
        resultado = f"{home_score}-{away_score}"
    else:
        resultado = "Pte. Jugar" 

    # CONVERSIÓN DE FECHA Y HORA (Timestamp Unix)
    timestamp = match_data.get('startTimestamp') 
    fecha_partido = None
    hora_partido = None
    
    if timestamp:
        try:
            # Convertir el timestamp a un objeto datetime
            # Los timestamps de Sofascore suelen estar en segundos
            dt_object = datetime.datetime.fromtimestamp(timestamp)
            
            # Formatear la fecha y hora
            fecha_partido = dt_object.strftime("%d/%m/%Y") 
            hora_partido = dt_object.strftime("%H:%M") 
        except (ValueError, TypeError):
            pass # Mantener None si la conversión falla

    return {
        "id": partido_id,
        "Local": local,
        "Visitante": visitante,
        "Estadio": estadio,
        "Capacidad": capacidad,
        "Latitud": latitud,
        "Longitud": longitud,
        "Resultado": resultado,
        "Fecha": fecha_partido,  
        "Hora": hora_partido    
    }


## 2. Flujo Principal y Exportación a Excel por Hojas

def process_and_export_matches(df_url: pd.DataFrame, start_jornada: int = 0, end_jornada: int = 14):
    """
    Procesa un DataFrame con URLs de Sofascore dentro de un rango de jornadas, 
    extrae la información y la guarda en un archivo Excel con hojas por jornada.
    
    :param df_url: DataFrame con las columnas 'Jornada', 'Partido' y 'URL'.
    :param start_jornada: La jornada inicial del rango a procesar (inclusive).
    :param end_jornada: La jornada final del rango a procesar (inclusive).
    """
    print(f"Iniciando procesamiento de jornadas {start_jornada} a {end_jornada}...")
    
    # 1. Preparación y Filtrado del DataFrame
    df_url["Jornada"] = pd.to_numeric(df_url["Jornada"], errors="coerce")
    df_url = df_url.dropna(subset=["Jornada", "URL"]).sort_values(["Jornada", "Partido"])

    # Aplicar el filtro de rango de jornadas
    df_url_filtrado = df_url[
        df_url["Jornada"].between(start_jornada, end_jornada)
    ].drop_duplicates(subset=["Jornada", "Partido", "URL"]).copy()

    partidos_info = []

    # 2. Bucle Dinámico sobre el DataFrame Filtrado
    print(f"URLs a procesar: {len(df_url_filtrado)}")
    
    for index, row in df_url_filtrado.iterrows():
        j = int(row["Jornada"])
        partido_num = int(row["Partido"]) if pd.notnull(row["Partido"]) else None
        url = row["URL"]

        try:
            # Llama a la función real de tu librería 'sofascore'
            match_data = sofascore.get_match_dict(url) 
            
            meta = extract_match_info_from_json(match_data)
            
            if meta:
                meta["Jornada"] = j
                meta["Partido"] = partido_num
                partidos_info.append(meta)

                print(f"Jornada {j} - Partido {partido_num} ({meta['Fecha']} {meta['Hora']}): "
                      f"{meta['Local']} vs {meta['Visitante']} | Rdo: {meta['Resultado']}")
            
        except Exception as e:
            print(f"Error al procesar URL {url} de Jornada {j} - Partido {partido_num}: {e}")
        
    # 3. Conversión a DataFrame y exportación por hojas
    df_meta = pd.DataFrame(partidos_info)

    if not df_meta.empty:
        grouped_by_jornada = df_meta.groupby('Jornada')
        output_filename = f"jornadas_{start_jornada:02d}_{end_jornada:02d}_info.xlsx"
        
        # Columnas en el orden solicitado (incluyendo Fecha y Hora)
        column_order = [
            "Jornada", "Partido", "id", "Fecha", "Hora", 
            "Local", "Visitante", "Resultado",
            "Estadio", "Capacidad", "Latitud", "Longitud"
        ]
        
        with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
            for jornada, df_jornada in grouped_by_jornada:
                sheet_name = f'Jornada {jornada}'
                
                df_jornada[column_order].to_excel(
                    writer, 
                    sheet_name=sheet_name, 
                    index=False
                )

        print(f"\nProceso completado. Guardado en: {output_filename} (con una hoja por jornada)")
    else:
        print("\n No se pudieron extraer datos. No se creó el archivo Excel.")
        
# --- EJEMPLO DE USO (REEMPLAZAR CON TU EJECUCIÓN REAL) ---
# 
# Si tu DataFrame df_url está cargado, puedes llamarla así:
process_and_export_matches(df_url, start_jornada=0, end_jornada=14)

Iniciando procesamiento de jornadas 0 a 14...
URLs a procesar: 140
✅ Jornada 1 - Partido 1 (15/08/2025 19:00): Girona FC vs Rayo Vallecano | Rdo: 1-3
✅ Jornada 1 - Partido 2 (15/08/2025 21:30): Villarreal vs Real Oviedo | Rdo: 2-0
✅ Jornada 1 - Partido 3 (16/08/2025 19:30): Mallorca vs Barcelona | Rdo: 0-3
✅ Jornada 1 - Partido 4 (16/08/2025 21:30): Deportivo Alavés vs Levante UD | Rdo: 2-1
✅ Jornada 1 - Partido 5 (16/08/2025 21:30): Valencia vs Real Sociedad | Rdo: 1-1
✅ Jornada 1 - Partido 6 (17/08/2025 17:00): Celta Vigo vs Getafe | Rdo: 0-2
✅ Jornada 1 - Partido 7 (17/08/2025 19:30): Athletic Club vs Sevilla | Rdo: 3-2
✅ Jornada 1 - Partido 8 (17/08/2025 21:30): Espanyol vs Atlético Madrid | Rdo: 2-1
✅ Jornada 1 - Partido 9 (18/08/2025 21:00): Elche vs Real Betis | Rdo: 1-1
✅ Jornada 1 - Partido 10 (19/08/2025 21:00): Real Madrid vs Osasuna | Rdo: 1-0
✅ Jornada 2 - Partido 1 (22/08/2025 21:30): Real Betis vs Deportivo Alavés | Rdo: 1-0
✅ Jornada 2 - Partido 2 (23/08/2025 17:00): Ma

In [4]:
sofascore.scrape_player_match_stats('https://www.sofascore.com/es-la/football/match/girona-fc-rayo-vallecano/tgbsoKj#id:14082854').head()

Running


,name,slug,shortName,position,jerseyNumber,height,userCount,gender,id,country,...,penaltyFaced,saves,keeperSaveValue,goalkeeperValueNormalized,totalKeeperSweeper,accurateKeeperSweeper,bigChanceMissed,penaltyWon,teamName,captain
0,Paulo Gazzaniga,paulo-gazzaniga,P. Gazzaniga,G,13,196.0,1895,M,164343,"{'alpha2': 'AR', 'alpha3': 'ARG', 'name': 'Arg...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Girona FC,NaN
1,Arnau Martínez,arnau-martinez,A. Martínez,D,4,181.0,1588,M,1084081,"{'alpha2': 'ES', 'alpha3': 'ESP', 'name': 'Spa...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Girona FC,True
2,David López,david-lopez,D. López,D,5,185.0,575,M,135116,"{'alpha2': 'ES', 'alpha3': 'ESP', 'name': 'Spa...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Girona FC,NaN
3,Ladislav Krejčí,ladislav-krejci,L. Krejčí,D,17,191.0,2129,M,856250,"{'alpha2': 'CZ', 'alpha3': 'CZE', 'name': 'Cze...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Girona FC,NaN
4,Daley Blind,daley-blind,D. Blind,D,17,180.0,2705,M,44864,"{'alpha2': 'NL', 'alpha3': 'NLD', 'name': 'Net...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Girona FC,NaN


In [ ]:
# SCRAPEO JUGADORES STAST 
import pandas as pd
import re
import xlsxwriter 
from pandas import json_normalize # Para aplanar la columna 'country'
import time 
# Importa tu librería sofascore aquí.


def process_and_export_player_stats_by_jornada(df_url: pd.DataFrame, sofascore_lib, start_jornada: int = 0, end_jornada: int = 14):
    """
    Procesa un rango de jornadas desde el df_url, extrae estadísticas de jugadores, 
    añade ID, Jornada y Partido, y exporta a un Excel con hojas separadas por Jornada.

    :param df_url: DataFrame con las URLs, Jornada y Partido.
    :param sofascore_lib: Módulo que contiene la función scrape_player_match_stats.
    :param start_jornada: Jornada inicial a procesar (inclusiva).
    :param end_jornada: Jornada final a procesar (inclusiva).
    """
    print(f"Iniciando extracción y exportación de Estadísticas de Jugadores (Jornadas {start_jornada} a {end_jornada})...")
    
    # 1. Preparación y Filtrado del DataFrame (usando df_url)
    df_url["Jornada"] = pd.to_numeric(df_url["Jornada"], errors="coerce")
    df_url = df_url.dropna(subset=["Jornada", "URL"]).sort_values(["Jornada", "Partido"])

    df_url_filtrado = df_url[
        df_url["Jornada"].between(start_jornada, end_jornada)
    ].drop_duplicates(subset=["Jornada", "Partido", "URL"]).copy()

    all_player_stats_dfs = []

    # 2. Bucle Dinámico y Extracción de Estadísticas
    for index, row in df_url_filtrado.iterrows():
        j = int(row["Jornada"])
        partido_num = int(row["Partido"]) if pd.notnull(row["Partido"]) else None
        url = row["URL"]
        
        # Extracción de ID del partido (último número después de #id:)
        match_id_match = re.search(r"id:(\d+)", url)
        partido_id = match_id_match.group(1) if match_id_match else None

        try:
            #  Llamada a la función que devuelve el DataFrame de estadísticas por jugador
            player_stats_df = sofascore_lib.scrape_player_match_stats(url)
            
            if isinstance(player_stats_df, pd.DataFrame) and not player_stats_df.empty:
                
                # 3. Normalización de la columna anidada 'country'
                if 'country' in player_stats_df.columns:
                    country_normalized = json_normalize(player_stats_df['country'], sep='_')
                    # Prefijo para evitar colisiones
                    country_normalized = country_normalized.add_prefix("country_")
                    
                    player_stats_df = pd.concat(
                        [player_stats_df.drop('country', axis=1).reset_index(drop=True),
                         country_normalized.reset_index(drop=True)],
                        axis=1
                    )
                
                # 4. CONCATENAR ID, JORNADA, PARTIDO (al DataFrame de estadísticas)
                player_stats_df["id"] = partido_id
                player_stats_df["Jornada"] = j
                player_stats_df["Partido"] = partido_num

                # 🔧 Resetear índice y eliminar columnas duplicadas
                player_stats_df = player_stats_df.reset_index(drop=True)
                player_stats_df = player_stats_df.loc[:, ~player_stats_df.columns.duplicated()]
                
                all_player_stats_dfs.append(player_stats_df)
                
                print(f" Jornada {j} - Partido {partido_num}: Estadísticas de {len(player_stats_df)} jugadores extraídas.")
            
            else:
                print(f" Jornada {j} - Partido {partido_num}: No hay datos de estadísticas de jugadores disponibles.")
            
        except Exception as e:
            print(f" Error al procesar estadísticas de jugadores para URL {url}: {e}")
            
    # 5. Consolidación y Exportación
    if not all_player_stats_dfs:
        print("\n No se pudieron obtener estadísticas de jugadores para ninguna URL. No se creará el Excel.")
        return

    df_stats_jugadores_raw = pd.concat(all_player_stats_dfs, ignore_index=True)
    # 🔧 Asegurar columnas únicas
    df_stats_jugadores_raw = df_stats_jugadores_raw.loc[:, ~df_stats_jugadores_raw.columns.duplicated()]

    # 6. Exportación a Excel con Hojas por Jornada
    output_stats = f"estadisticas_jugadores_por_jornada_{start_jornada:02d}_{end_jornada:02d}.xlsx"
    grouped_by_jornada = df_stats_jugadores_raw.groupby('Jornada')

    # Definir el orden de las columnas: Contexto primero
    context_columns = ["Jornada", "Partido", "id"]
    
    try:
        with pd.ExcelWriter(output_stats, engine='xlsxwriter') as writer:
            for jornada, df_jornada in grouped_by_jornada:
                sheet_name = f'Jornada {int(jornada)}'
                
                # Asegurar que las columnas de contexto estén al inicio
                all_cols = context_columns + [col for col in df_jornada.columns if col not in context_columns]
                
                df_jornada[all_cols].to_excel(
                    writer, 
                    sheet_name=sheet_name, 
                    index=False
                )

        print(f"\n Proceso completado. Exportado en: {output_stats} (Hoja por Jornada)")
    except Exception as e:
        print(f"S Error al escribir el archivo Excel: {e}")


# ==============================================================================
# 📢 CÓDIGO DE EJECUCIÓN (Ejemplo)
# 
# Para usarlo, simplemente llama a la función con tu DataFrame y tu librería:
process_and_export_player_stats_by_jornada(df_url, sofascore, start_jornada=1, end_jornada=14)
# ==============================================================================

Iniciando extracción y exportación de Estadísticas de Jugadores (Jornadas 1 a 14)...
Running
 Jornada 1 - Partido 1: Estadísticas de 44 jugadores extraídas.
 Jornada 1 - Partido 2: Estadísticas de 46 jugadores extraídas.
 Jornada 1 - Partido 3: Estadísticas de 46 jugadores extraídas.
 Jornada 1 - Partido 4: Estadísticas de 45 jugadores extraídas.
 Jornada 1 - Partido 5: Estadísticas de 46 jugadores extraídas.
 Jornada 1 - Partido 6: Estadísticas de 41 jugadores extraídas.
 Jornada 1 - Partido 7: Estadísticas de 45 jugadores extraídas.
 Jornada 1 - Partido 8: Estadísticas de 46 jugadores extraídas.
 Jornada 1 - Partido 9: Estadísticas de 45 jugadores extraídas.
 Jornada 1 - Partido 10: Estadísticas de 44 jugadores extraídas.
 Jornada 2 - Partido 1: Estadísticas de 43 jugadores extraídas.
 Jornada 2 - Partido 2: Estadísticas de 46 jugadores extraídas.
 Jornada 2 - Partido 3: Estadísticas de 45 jugadores extraídas.
 Jornada 2 - Partido 4: Estadísticas de 43 jugadores extraídas.
 Jornada 2

In [ ]:
# SCRAPEO ESCUDOS 


from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd

# Configuración del navegador (Chrome en este ejemplo)
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Ejecutar sin abrir ventana
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(options=options)

# URL de LaLiga en Sofascore
url = "https://www.sofascore.com/es/torneo/futbol/spain/laliga/8#id:77559"
driver.get(url)

# Espera para que cargue el contenido dinámico
time.sleep(5)

# Buscar todas las imágenes de equipos
imagenes = driver.find_elements(By.CSS_SELECTOR, "img.Img")

equipos = []
for img in imagenes:
    nombre = img.get_attribute("alt")
    logo = img.get_attribute("src")
    if nombre and logo:
        equipos.append({"Equipo": nombre, "Logo": logo})

driver.quit()

# Mostrar resultados
df = pd.DataFrame(equipos)
print(df)

# Guardar en CSV si lo necesitas
df.to_excel("escudos_equipo.xlsx", index=False)

              Equipo                                               Logo
0             LaLiga  https://img.sofascore.com/api/v1/unique-tourna...
1              Spain   https://img.sofascore.com/api/v1/country/ES/flag
2          Barcelona   https://img.sofascore.com/api/v1/team/2817/image
3        Real Madrid   https://img.sofascore.com/api/v1/team/2829/image
4         Villarreal   https://img.sofascore.com/api/v1/team/2819/image
5    Atlético Madrid   https://img.sofascore.com/api/v1/team/2836/image
6         Real Betis   https://img.sofascore.com/api/v1/team/2816/image
7           Espanyol   https://img.sofascore.com/api/v1/team/2814/image
8      Athletic Club   https://img.sofascore.com/api/v1/team/2825/image
9             Getafe   https://img.sofascore.com/api/v1/team/2859/image
10  Deportivo Alavés   https://img.sofascore.com/api/v1/team/2885/image
11    Rayo Vallecano   https://img.sofascore.com/api/v1/team/2818/image
12             Elche   https://img.sofascore.com/api/v1/team/284

In [6]:
equipos

[]